# kNN + Metrics + Scaling Effects

**Week 4 · Lesson 4** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week4/lectures/04_knn_metrics_scaling.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 3.5 (linear regression vs. K-nearest neighbors) where relevant.*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** can we price a home the way an appraiser does — by finding
the most comparable recent sales ("comps") and averaging what they sold for — and
does the answer depend on how we measure "comparable"?

**Why it matters:** Week 3 priced homes with a formula. Comps are how pricing
actually gets explained to a seller: "these 13 similar homes sold for about this
much." That is kNN regression exactly — Lesson 3's "average the neighbors'
targets." Lessons 1–3's choices (which distance metric, whether to standardize,
how many comps) have been abstract until now. Here they decide which homes count
as comparable, and so what price the model quotes.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset as Week 3 — **Ames Housing**, one row per home sale (2006–2010,
Ames, Iowa), with `SalePrice` in dollars as the target. `load_housing()` in
`gbus564_coursework.utils.course_data` loads and cleans it the same way Week 3 did.
This notebook uses the same 8 predictors Week 3 Lesson 2 hand-picked, so kNN's
price estimates can be compared directly against that lesson's linear regression.

Why not all 35 numeric columns? For kNN, more isn't better. Once features are
standardized, every column gets an equal say in the distance, including ones
that barely relate to price (`PoolArea`, `MiscVal`, the month a home sold), and
each one adds noise to what "comparable" means. Week 3's lasso could learn to
ignore columns like that; kNN can't — it weighs whatever it's given. Tuned the
same way as below, kNN on all 35 numeric columns prices slightly *worse* than on
these 8 (CV RMSE about \$30.9k vs. \$30.3k). That's Lesson 2's curse of
dimensionality on real data: for kNN, choosing the features is part of the model.

In [ ]:
from gbus564_coursework.utils.course_data import load_housing

df = load_housing()
predictors = ["GrLivArea", "TotRmsAbvGrd", "OverallQual", "YearBuilt",
              "TotalBsmtSF", "GarageCars", "LotArea", "FullBath"]
df[predictors + ["SalePrice"]].head()

In [ ]:
df[predictors + ["SalePrice"]].describe()

### Data Prep

kNN's distance *is* the raw-difference sum Lesson 1 introduced — unlike Week 5's
trees, which split on raw thresholds and don't care about scale, kNN cares about
nothing else. Look at the spreads in the table above: `LotArea` is measured in
square feet and varies by about 8,000 from home to home, while `OverallQual` is a
1–10 rating that varies by about 1.4. Summed raw, a 500 sq ft difference in lot
size outweighs the whole quality scale, top to bottom. **Standardizing** puts
every feature in units of its own typical spread, so a gap counts by how unusual
it is, not by how big its units are — exactly Lesson 2's lesson.

The cell below holds out the test set (same 80/20 split and seed as Week 3, so
the test homes are the same), defines the scaler, and then makes Lesson 2's point
concrete. For *every* training home, it finds the 10 nearest neighbors two ways —
on the raw features and on the standardized ones — and asks what those "comps"
actually have in common with the home.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

X, y = df[predictors], df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
print(f"Training homes: {len(X_train):,}   Test homes: {len(X_test):,}")

# Defined here, fit in Model (inside a Pipeline, on training rows only).
scale = StandardScaler()

# For the demo below only: the training rows raw, and standardized.
X_unscaled = X_train.to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X_train)

In [ ]:
K_DEMO = 10

def neighbor_profile(X_matrix):
    """Each home's K_DEMO nearest neighbors, and how closely they match it."""
    # kneighbors() with no argument queries every fitted home and leaves each one
    # out of its own neighbor list (safer than dropping the first column: some
    # homes have identical features, so they tie at distance 0).
    _, idx = NearestNeighbors(n_neighbors=K_DEMO).fit(X_matrix).kneighbors()

    # Which features the neighbor distances are actually made of (squared gaps).
    sq_diff = ((X_matrix[idx] - X_matrix[:, None, :]) ** 2).sum(axis=(0, 1))
    lot_share = sq_diff[predictors.index("LotArea")] / sq_diff.sum()

    def median_gap(values):
        values = np.asarray(values)
        return np.median(np.abs(values[idx] - values[:, None]))

    profile = pd.Series({
        "LotArea's share of the distance": f"{lot_share:.0%}",
        "median lot-size gap": f"{median_gap(X_train['LotArea']):,.0f} sq ft",
        "median quality gap (1–10)": f"{median_gap(X_train['OverallQual']):.0f}",
        "median year-built gap": f"{median_gap(X_train['YearBuilt']):.0f} years",
        "median sale-price gap": f"${median_gap(y_train):,.0f}",
    })
    return profile, idx

profile_unscaled, idx_unscaled = neighbor_profile(X_unscaled)
profile_scaled, idx_scaled = neighbor_profile(X_scaled)

overlap = np.mean([len(set(a) & set(b)) for a, b in zip(idx_unscaled, idx_scaled)])
print(f"Average neighbors in common, unscaled vs. standardized: {overlap:.1f} of {K_DEMO}")
pd.DataFrame({"unscaled": profile_unscaled, "standardized": profile_scaled})

Standardizing changes *which homes count as comps* almost completely: on average,
only about 1 of a home's 10 neighbors survives the switch. Unscaled, the distance
is essentially all lot size — `LotArea` alone is over 99% of it — so a home's
"comps" are homes on a similar-sized lot that can be a full quality grade apart
and built in a different decade. That's backwards: of the 35 numeric columns in
this data, `LotArea` is only 21st by correlation with sale price, so the unscaled
model lets one of the *weakest* features decide who counts as comparable, purely
because it has the biggest units. Standardized, every feature gets a say: the comps
match on quality and are typically built within a few years of each other, even
though their lots differ more. And the homes they pick sold for much closer to the
same price — which is the whole point of a comp.

One thing standardizing doesn't fix: `LotArea` is still the biggest single share
of the standardized distance. A few homes sit on enormous lots (the largest is over
200,000 sq ft, about 25 standard deviations above average), so their gaps stay huge
even in standardized units. Standardizing fixes *units*; it doesn't fix *skew*.

## 3. Model

*Baseline first — the reference every model must beat — then the model under
study.*

`DummyRegressor` (always quote the average training price) is the same Week 3
baseline every pricing model has to clear. The model under study is a kNN
regressor on the standardized features from the Data Prep step: it prices a home
at the average sale price of its k nearest training homes. Its two knobs — k and
the distance metric — are chosen next, by cross-validation, rather than left at
scikit-learn's defaults.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

### Model Selection

Two knobs, swept together: **k** (Lesson 3's bias-variance dial — how many comps
get averaged) and the **distance metric** (Lesson 1's choice of ruler — Euclidean,
Manhattan, or cosine). `GridSearchCV` cross-validates every (k, metric)
combination on the training data only, the same disciplined way Week 3 chose λ
for ridge/lasso — never by peeking at the test set. `algorithm="brute"` is set
not for speed, but because it's the only algorithm scikit-learn lets swap in
every metric Lesson 1 covered, including cosine.

The search is scored on **RMSE** in dollars, the same yardstick Week 3 used.
scikit-learn always *maximizes* a score, so it's passed as
`"neg_root_mean_squared_error"` (−RMSE) and flipped back for display.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": range(1, 52, 2),
    "knn__metric": ["euclidean", "manhattan", "cosine"],
}
knn_search = GridSearchCV(
    Pipeline([("scale", scale), ("knn", KNeighborsRegressor(algorithm="brute"))]),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
knn_search.fit(X_train, y_train)

print(f"Best combination: {knn_search.best_params_}")
print(f"Best CV RMSE: ${-knn_search.best_score_:,.0f}")

cv_results = pd.DataFrame(knn_search.cv_results_)
cv_results["cv_rmse"] = -cv_results["mean_test_score"]
best_per_metric = (
    cv_results.loc[cv_results.groupby("param_knn__metric")["cv_rmse"].idxmin()]
    [["param_knn__metric", "param_knn__n_neighbors", "cv_rmse"]]
    .sort_values("cv_rmse")
    .rename(columns={"param_knn__metric": "metric", "param_knn__n_neighbors": "best k",
                     "cv_rmse": "CV RMSE"})
)
best_per_metric.style.format({"CV RMSE": "${:,.0f}"})

In [ ]:
fig, ax = plt.subplots()
for metric, runs in cv_results.groupby("param_knn__metric"):
    ax.plot(runs["param_knn__n_neighbors"].astype(int), runs["cv_rmse"],
            marker="o", label=metric)
ax.set(xlabel="k (comps averaged)", ylabel="CV RMSE ($)",
       title="Cross-validated pricing error by k and distance metric")
ax.yaxis.set_major_formatter("${x:,.0f}")
ax.legend(title="metric")
plt.tight_layout()

**k is Lesson 3's bias-variance dial, made visible.** With k = 1, a home is priced
at whatever its single closest sale went for — one comp's quirks become the
estimate, and the error is high. Averaging more comps smooths those quirks out
and the error falls fast, bottoming out around a dozen comps. Past that, the
average starts pulling in homes that aren't really comparable, and the error
creeps back up. The best k sits in the middle of the grid, not at an edge.

**The metric matters here too:**

- **Manhattan edges out Euclidean** — Euclidean squares each gap, so one big
  mismatch (a much larger lot, a much bigger basement) can dominate the whole
  distance. Manhattan adds gaps as they are, so a comp that matches well on most
  features isn't thrown out over one. That's Lesson 1's case for Manhattan when
  one feature's outliers shouldn't dominate.
- **Cosine is clearly worst** — and for a reason worth remembering. Cosine
  compares only a vector's *direction*, ignoring its length. On standardized
  features, a slightly-better-than-average home (+0.5σ on everything) and a
  mansion (+3σ on everything) point the same way, so cosine calls them
  comparable. But for pricing, *how far* above average a home is is exactly what
  sets its price. Cosine earns its keep when length is noise you want to ignore —
  Lesson 5's purchase vectors, where one customer buys 3 things and another
  buys 300.

## 4. Evaluate

*Score against the baseline, not in isolation.*

Four models on the held-out test homes: the baseline; Week 3 Lesson 2's linear
regression on the same 8 predictors and the same test homes; the tuned kNN; and
one more that isolates scaling's contribution on its own — kNN on the *raw*
features, given its own search over the same (k, metric) grid so the comparison is
fair. RMSE is what the searches optimized; MAE rides along as the "typical miss"
in plain dollars.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

tuned_knn = knn_search.best_estimator_
linear = LinearRegression().fit(X_train, y_train)

# Isolate scaling's effect: no StandardScaler, but its own search over the same
# (k, metric) grid — the raw-feature model may prefer a different k or metric,
# so reusing the standardized model's pick would handicap it.
unscaled_search = GridSearchCV(
    Pipeline([("knn", KNeighborsRegressor(algorithm="brute"))]),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
).fit(X_train, y_train)
unscaled_knn = unscaled_search.best_estimator_
print(f"Standardized pick: {knn_search.best_params_}   CV RMSE ${-knn_search.best_score_:,.0f}")
print(f"Unscaled pick:     {unscaled_search.best_params_}   CV RMSE ${-unscaled_search.best_score_:,.0f}")

models = {
    "Baseline (mean price)": baseline,
    "Linear regression, same 8 predictors (Week 3)": linear,
    "Tuned kNN, unscaled features": unscaled_knn,
    "Tuned kNN, standardized features": tuned_knn,
}
pd.DataFrame({
    name: {
        "test RMSE": root_mean_squared_error(y_test, model.predict(X_test)),
        "test MAE": mean_absolute_error(y_test, model.predict(X_test)),
    }
    for name, model in models.items()
}).T.style.format("${:,.0f}")

Here scaling pays off, clearly and consistently. Both searches pick Manhattan
with 13 comps, so the only difference between the last two rows is
standardizing — and it cuts the pricing error by about a third (test RMSE about
\$45k unscaled vs. \$31k standardized), with cross-validation telling the same
story. The Data Prep check explains why: unscaled, "comparable" means "similar lot
size" and almost nothing else, and lot size alone is a poor guide to price.

The standardized kNN also beats Week 3 Lesson 2's linear regression on the same 8
predictors (\$36,128 test RMSE). It doesn't need a straight-line relationship
between each feature and price — it just finds similar homes and averages. Week 3
Lesson 9's lasso, with all 285 encoded columns, still does better (\$27,982), so
kNN's edge here is over a formula with the same information, not over every model.

In [ ]:
pred = tuned_knn.predict(X_test)

fig, ax = plt.subplots()
ax.scatter(y_test, pred, alpha=0.4)
lims = [0, max(y_test.max(), pred.max()) * 1.05]
ax.plot(lims, lims, color="gray", linestyle="--", label="perfect pricing")
ax.set(xlim=lims, ylim=lims, xlabel="Actual sale price", ylabel="kNN price (average of comps)",
       title="Tuned kNN on the test homes")
ax.xaxis.set_major_formatter("${x:,.0f}")
ax.yaxis.set_major_formatter("${x:,.0f}")
ax.legend()
plt.tight_layout()

Most homes sit close to the line. The visible pattern is at the top end: the most
expensive homes fall *below* it. Lesson 3's warning — kNN can't predict past what
it has seen — shows up here in a softer form: a home's estimate is an average of
its comps, and the priciest homes have few comps that are as expensive as they
are, so the average gets pulled down. (Week 3's linear regression underprices
these homes too, by even more, so this is a hard segment for both — but for kNN
the reason is built in.)

The one point far *above* the line is a different story: a 4,676 sq ft home that
sold for about \$185k as a "Partial" sale — one of a handful of very large homes
in this data that the dataset's author flags as unusual sales, not market prices.
Its comps are other big, high-quality homes, so kNN prices it like one. That's
the model doing its job on a sale that isn't telling the truth about the home.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

kNN has no coefficients (Week 3) and no splitting rules to trace (Week 5) — its
explanation is Lesson 3's retrieval reframe made literal: the model's reasoning
for any one home **is** its list of comps. The cell below picks a typical test
home (the one whose pricing error is closest to the median error) and shows them.

In [ ]:
errors = np.abs(pred - y_test)
query = int((errors - errors.median()).abs().idxmin())  # a typical miss, not a cherry-pick
print(f"Test home: actual sale price ${y_test[query]:,.0f}, "
      f"kNN estimate ${pred[query]:,.0f}\n")
display(X_test.iloc[[query]])

query_scaled = tuned_knn.named_steps["scale"].transform(X_test.iloc[[query]])
distances, comp_idx = tuned_knn.named_steps["knn"].kneighbors(query_scaled)

comps = X_train.iloc[comp_idx[0]][["GrLivArea", "OverallQual", "YearBuilt", "LotArea"]].copy()
comps.insert(0, "distance", distances[0].round(2))
comps["SalePrice"] = y_train.iloc[comp_idx[0]].values
comps.style.format({"distance": "{:.2f}", "SalePrice": "${:,.0f}", "LotArea": "{:,}", "GrLivArea": "{:,}"})

Every row above is a real training sale, ranked by how close its (standardized)
profile sits to the test home's, and the estimate is simply the average of their
`SalePrice` column. That table *is* the model's explanation, not a summary of one —
and it's the same form an appraiser's report takes. A seller who pushes back on
the price can see exactly which sales it came from, and argue about specific comps
("that one has no garage") instead of a formula.

## 6. Recommend

**Recommendation:** use the tuned kNN as a **comps engine** for pricing
conversations: quote its estimate together with its comps table. On the same 8
features Week 3 used, it prices more accurately than Week 3's linear regression
(test RMSE about \$31k vs. \$36k), and every estimate comes with its own evidence.
Two settings are not optional: **standardize the features first** (it cut pricing
error by about a third here; without it, "comparable" just means "similar lot
size"), and **use Manhattan distance, not cosine** (for pricing, how far above
average a home is matters, and cosine throws that away).

**Caveats / next steps:** kNN pulls the most expensive (and cheapest) homes toward
the middle, since an estimate can only be as extreme as the comps available —
treat its quotes for unusual homes as a floor (or ceiling), not an answer. And
standardizing doesn't fix skew: `LotArea` still dominates the distance for homes
on very large lots. Lesson 5 takes the same "find the most similar" machinery
somewhere new: instead of "which sales are like this home," "which products does a
customer like this one tend to buy."